# Lightweight Local SLM LoRA Fine-Tuning (16 GB MacBook Pro Edition)

Optimized for Apple Silicon Macs with 16 GB Unified Memory:
- Uses `mlx-community/Qwen2.5-1.5B-Instruct-4bit` (~900 MB footprint)
- Caps `--max-seq-length 192` with `--batch-size 2` and `--grad-checkpoint` to prevent swap
- Evaluates on a stratified 150-row holdout sample (~15 seconds) with explicit Metal cache cleanup (`mx.clear_cache()`)

In [15]:
# 1. Install MLX-LM & Dependencies into Active Kernel
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "mlx-lm", "pandas", "scikit-learn"], check=True)


[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


CompletedProcess(args=['/Users/aloksingh/git/home-category-api/.venv-slm/bin/python', '-m', 'pip', 'install', '-q', '-U', 'mlx-lm', 'pandas', 'scikit-learn'], returncode=0)

In [16]:
# 2. Set common ENV Variables
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]  # Resolves to /Users/aloksingh/git/home-category-api
CSV_PATH = os.path.expanduser("~/Downloads/Expense Sheet - Categorized.csv")
CORRECTIONS_PATH = PROJECT_ROOT / "data" / "corrections.csv"
LORA_DATA_DIR = PROJECT_ROOT / "training" / "lora" / "lora_data"
ADAPTER_DIR = PROJECT_ROOT / "training" / "lora" / "adapters" / "expense-lora-lite"
FUSED_MODEL_DIR = PROJECT_ROOT / "training" / "lora" / "model" / "expense-lora-1.5b"

BASE_MODEL = "mlx-community/Qwen2.5-1.5B-Instruct-4bit"

In [17]:
# 3. Prepare & Clean Stratified ChatML Dataset
import json
import re
import urllib.parse
import pandas as pd
from sklearn.model_selection import train_test_split


DIAGNOSTIC_PATTERN = (
    r'\b(?:blood\s*test|lab\s*test|diagnostic|x[\s-]*ray|mri|ct\s*scan|'
    r'ultrasound|sonography|ecg|echo|pathology|biopsy|cbc|thyroid|'
    r'lipid\s*profile|vitamin\s*[bd]|checkup|health\s*check|'
    r'thyrocare|lal\s*pathlabs|aarthi\s*scans|metropolis|orange\s*health|healthians)\b'
)

def clean_text(text: str) -> str:
    text = urllib.parse.unquote(str(text))
    text = re.sub(r'has been debited from account.*?to VPA', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'spent on your SBI Credit Card ending \d+ at', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'UPI transaction reference no.*', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'To check your available balance.*', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'[\d_]|[^\w\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

df = pd.read_csv(CSV_PATH)
df = df.rename(columns={'head': 'Head', 'category': 'Category'})
df = df[~df['Head'].str.strip().str.lower().str.contains(r'dummy|^t\s*est|^tes$', regex=True, na=False)]
df = df[pd.to_numeric(df['amount'], errors='coerce') > 1.0].copy()

if CORRECTIONS_PATH.exists():
    corr_df = pd.read_csv(CORRECTIONS_PATH)
    corr_df = corr_df.rename(columns={'head': 'Head', 'category': 'Category'})
    corr_df = corr_df[
        ~corr_df['Head'].str.strip().str.lower().str.contains(r'dummy|^t\s*est|^tes$', regex=True, na=False)
    ].copy()
    corr_df['amount'] = pd.to_numeric(corr_df['amount'], errors='coerce').fillna(0.0)
    df = pd.concat([df, corr_df, corr_df], ignore_index=True)

df['Category'] = df['Category'].replace({'Shopping': 'Accessories', 'Househelp': 'House Help'})
combined_text = (df['Head'].fillna('') + ' ' + df['comment'].fillna('')).str.lower()
df.loc[combined_text.str.contains(DIAGNOSTIC_PATTERN, regex=True, na=False), 'Category'] = 'Medical'

counts = df['Category'].value_counts()
df = df[df['Category'].isin(counts[counts >= 5].index)].copy()

VALID_CATEGORIES = sorted(df['Category'].unique().tolist())
SYSTEM_PROMPT = (
    "Classify the Indian household expense into one valid category: "
    + ", ".join(VALID_CATEGORIES)
    + ". Reply with ONLY the category name."
)

def format_user_prompt(head: str, comment: str, amount: float) -> str:
    c_clean = clean_text(comment) if pd.notna(comment) and str(comment).strip() else "None"
    h_clean = str(head).strip()
    amt_str = f"₹{float(amount):.2f}" if float(amount) > 1.0 else "Unknown"
    return f"Head: {h_clean} | Comment: {c_clean} | Amount: {amt_str}"

def to_chatml_record(row) -> dict:
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": format_user_prompt(row['Head'], row['comment'], row['amount'])},
            {"role": "assistant", "content": str(row['Category']).strip()},
        ]
    }

train_df, valid_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df['Category'])

LORA_DATA_DIR.mkdir(parents=True, exist_ok=True)
for split_name, split_df in [("train", train_df), ("valid", valid_df)]:
    out_path = LORA_DATA_DIR / f"{split_name}.jsonl"
    with open(out_path, "w", encoding="utf-8") as f:
        for _, row in split_df.iterrows():
            f.write(json.dumps(to_chatml_record(row), ensure_ascii=False) + "\n")

print(f"Project Root: {PROJECT_ROOT}")
print(f"Prepared {len(train_df)} train and {len(valid_df)} validation rows across {len(VALID_CATEGORIES)} categories.")

Project Root: /Users/aloksingh/git/home-category-api
Prepared 6369 train and 1125 validation rows across 23 categories.


In [18]:
# 3. Memory-Safe LoRA Fine-Tuning (1.5B 4-bit, batch_size=2, max_seq_length=192, grad_checkpoint)
import subprocess
import sys

cmd = [
    sys.executable, "-m", "mlx_lm", "lora",
    "--model", BASE_MODEL,
    "--train",
    "--data", str(LORA_DATA_DIR),
    "--iters", "350",
    "--batch-size", "2",
    "--max-seq-length", "192",
    "--num-layers", "4",
    "--grad-checkpoint",
    "--learning-rate", "1.5e-4",
    "--steps-per-eval", "100",
    "--val-batches", "10",
    "--adapter-path", str(ADAPTER_DIR),
]
subprocess.run(cmd, check=True)

Loading pretrained model


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 2520.28it/s]


Loading datasets
Training
╭─ mlx_lm.lora ────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│    model           mlx-community/Qwen2.5-1.5B-Instruct-4bit                                                          │
│    type            lora · 4 layers · rank 8                                                                          │
│    dataset         /Users/aloksingh/git/home-category-api/training/lora/lora_data                                    │
│    optimizer       adam · lr 1.5e-04                                                                                 │
│    batch · iters   2 · 350                                                                                           │
│    max seq         192                                                                                               │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
Traina

CompletedProcess(args=['/Users/aloksingh/git/home-category-api/.venv-slm/bin/python', '-m', 'mlx_lm', 'lora', '--model', 'mlx-community/Qwen2.5-1.5B-Instruct-4bit', '--train', '--data', '/Users/aloksingh/git/home-category-api/training/lora/lora_data', '--iters', '350', '--batch-size', '2', '--max-seq-length', '192', '--num-layers', '4', '--grad-checkpoint', '--learning-rate', '1.5e-4', '--steps-per-eval', '100', '--val-batches', '10', '--adapter-path', '/Users/aloksingh/git/home-category-api/training/lora/adapters/expense-lora-lite'], returncode=0)

In [19]:
# 4. Fast Stratified Holdout Evaluation (~15 Seconds, Low Memory)
import json
import random
from collections import defaultdict
import mlx.core as mx
from mlx_lm import load, generate
from sklearn.metrics import classification_report, accuracy_score

model, tokenizer = load(BASE_MODEL, adapter_path=str(ADAPTER_DIR))

with open(LORA_DATA_DIR / "valid.jsonl", "r", encoding="utf-8") as f:
    valid_records = [json.loads(line) for line in f]

# Stratified sample: up to 7 rows per category (~140 rows total)
random.seed(42)
by_cat = defaultdict(list)
for rec in valid_records:
    by_cat[rec["messages"][2]["content"]].append(rec)

eval_sample = []
for cat, items in by_cat.items():
    eval_sample.extend(random.sample(items, min(len(items), 7)))

print(f"Evaluating stratified sample of {len(eval_sample)} / {len(valid_records)} validation rows...")

y_true, y_pred = [], []
hallucinated = []

for idx, rec in enumerate(eval_sample, start=1):
    messages = rec["messages"][:2]
    expected = rec["messages"][2]["content"]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    pred = generate(model, tokenizer, prompt=prompt, max_tokens=6, verbose=False).strip()

    y_true.append(expected)
    y_pred.append(pred)
    if pred not in VALID_CATEGORIES:
        hallucinated.append((messages[1]["content"], expected, pred))

    if idx % 20 == 0:
        mx.clear_cache()

print(f"\nStratified Holdout Accuracy: {accuracy_score(y_true, y_pred):.2%}")
print(f"Out-of-vocabulary predictions: {len(hallucinated)}")
print(classification_report(y_true, y_pred, zero_division=0))

/Users/aloksingh/git/home-category-api/.venv-slm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 1604.01it/s]


Evaluating stratified sample of 147 / 1125 validation rows...

Stratified Holdout Accuracy: 73.47%
Out-of-vocabulary predictions: 6
                   precision    recall  f1-score   support

      Accessories       0.78      1.00      0.88         7
       Appliances       0.00      0.00      0.00         3
        Baby Care       1.00      0.43      0.60         7
          Battery       0.00      0.00      0.00         0
            Bills       1.00      1.00      1.00         7
      Celebration       1.00      0.29      0.44         7
        Education       0.83      0.71      0.77         7
      Electronics       1.00      0.57      0.73         7
        Emergency       0.00      0.00      0.00         0
    Entertainment       0.00      0.00      0.00         2
          Fitness       0.00      0.00      0.00         7
     Food Outside       0.78      1.00      0.88         7
             Fuel       1.00      1.00      1.00         7
        Furniture       1.00      0.50   

In [20]:
# 5. Edge-Case Regression Sanity Check
edge_cases = [
    ("Dishwasher", "", 459.0, "Grocery"),
    ("Dishwasher", "INR 32472 at Flipkart In on 10-10-2024", 32472.0, "Appliances"),
    ("Hotter recho", "149.00 debited to VPA hotstaronline@ybl JioHotstar", 149.0, "Bills"),
    ("ऑटो", "", 107.0, "Travel"),
    ("Detergent powder", "", 666.0, "Grocery"),
    ("Lunch", "COMPASS INDIA FOOD SERVICES PR", 630.0, "Food Outside"),
    ("Blood test", "Thyrocare", 1200.0, "Medical"),
]

for head, comment, amount, expected in edge_cases:
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": format_user_prompt(head, comment, amount)},
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    pred = generate(model, tokenizer, prompt=prompt, max_tokens=6, verbose=False).strip()
    status = "PASS" if pred == expected else "FAIL"
    print(f"{status} {head:<20} (₹{amount:>8.2f}) -> Expected: {expected:<14} | Actual: {pred}")

# Free model from memory before fusing on disk
del model, tokenizer
mx.clear_cache()

FAIL Dishwasher           (₹  459.00) -> Expected: Grocery        | Actual: Maintenance
FAIL Dishwasher           (₹32472.00) -> Expected: Appliances     | Actual: Household
PASS Hotter recho         (₹  149.00) -> Expected: Bills          | Actual: Bills
PASS ऑटो                  (₹  107.00) -> Expected: Travel         | Actual: Travel
PASS Detergent powder     (₹  666.00) -> Expected: Grocery        | Actual: Grocery
PASS Lunch                (₹  630.00) -> Expected: Food Outside   | Actual: Food Outside
PASS Blood test           (₹ 1200.00) -> Expected: Medical        | Actual: Medical


In [21]:
# 6. Fuse LoRA Adapter into Standalone Model Directory
import subprocess
import sys

cmd = [
    sys.executable, "-m", "mlx_lm", "fuse",
    "--model", BASE_MODEL,
    "--adapter-path", str(ADAPTER_DIR),
    "--save-path", str(FUSED_MODEL_DIR),
]
subprocess.run(cmd, check=True)
print(f"Fused standalone 1.5B SLM saved to {FUSED_MODEL_DIR}")

Loading pretrained model


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 2135.11it/s]


Fused standalone 1.5B SLM saved to /Users/aloksingh/git/home-category-api/training/lora/model/expense-lora-1.5b


In [25]:
# 7. Dequantize, Fuse LoRA Adapter, Build Qwen2.5 ChatML Modelfile, and Import into Ollama
import shutil
import subprocess
import sys

FUSED_HF_DIR = PROJECT_ROOT / "training" / "lora" / "model" / "expense-lora-1.5b-hf"
MODELFILE_PATH = PROJECT_ROOT / "training" / "lora" / "model" / "Modelfile"
OLLAMA_MODEL_NAME = "expense-categorizer"

FUSED_HF_DIR.parent.mkdir(parents=True, exist_ok=True)

# 1. Fuse LoRA adapter and dequantize weights into HF safetensors for Ollama conversion
fuse_cmd = [
    sys.executable, "-m", "mlx_lm", "fuse",
    "--model", BASE_MODEL,               # mlx-community/Qwen2.5-1.5B-Instruct-4bit
    "--adapter-path", str(ADAPTER_DIR),  # adapters/expense-lora-lite
    "--save-path", str(FUSED_HF_DIR),
    "--dequantize",
]
subprocess.run(fuse_cmd, check=True)

# 2. Always point FROM at ./expense-slm-1.5b-hf so Ollama imports the newly fused weights
modelfile_content = f"""FROM ./{FUSED_HF_DIR.name}

TEMPLATE \"\"\"{{{{- if .System }}}}<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{ end }}}}{{{{- if .Prompt }}}}<|im_start|>user
{{{{ .Prompt }}}}<|im_end|>
{{{{ end }}}}<|im_start|>assistant
\"\"\"

PARAMETER stop "<|im_start|>"
PARAMETER stop "<|im_end|>"
PARAMETER temperature 0.0
PARAMETER num_predict 8

SYSTEM \"\"\"{SYSTEM_PROMPT}\"\"\"
"""
MODELFILE_PATH.write_text(modelfile_content, encoding="utf-8")
print(f"Wrote ChatML Modelfile pointing to ./{FUSED_HF_DIR.name}")

# 3. Quantize directly to Q4_K_M (~1.1 GB) and overwrite expense-categorizer in Ollama
subprocess.run(
    ["ollama", "create", OLLAMA_MODEL_NAME, "-f", str(MODELFILE_PATH), "-q", "Q4_K_M"],
    cwd=str(MODELFILE_PATH.parent),
    check=True,
)
print(f"Updated Ollama model: {OLLAMA_MODEL_NAME}")

# 4. Reclaim ~3 GB disk space once imported into Ollama
shutil.rmtree(FUSED_HF_DIR, ignore_errors=True)
print(f"Cleaned up temporary HF directory: {FUSED_HF_DIR}")


Loading pretrained model


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 2992.84it/s]


Dequantizing model


Wrote ChatML Modelfile pointing to ./expense-lora-1.5b-hf


gathering model components ⠙ gathering model components ⠹ gathering model components ⠹ gathering model components ⠼ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠇ gathering model components ⠏ gathering model components ⠋ gathering model components ⠙ gathering model components ⠹ gathering model components ⠸ gathering model components ⠼ gathering model components ⠴ gathering model components ⠦ gathering model components ⠧ gathering model components ⠇ gathering model components ⠇ gathering model components ⠋ gathering model components ⠙ gathering model components ⠹ gathering model components ⠸ gathering model components ⠼ gathering model components 
copying file sha256:86afa7d8ef2e622fb9a917c75008198e791eac225937c687c0910fc6e46f377e 100% 
copying file sha256:62028535ce9a2e3c00d376afc32f7342911a196f3dcb8977b9d4b1933874ca0b 100% 
copying file sha256:3fd169731d2cbde95e10bf356d66d5997fd885dd8dbb6fb4684da3f23b2585d8 100% 
cop

Updated Ollama model: expense-categorizer
Cleaned up temporary HF directory: /Users/aloksingh/git/home-category-api/training/lora/model/expense-lora-1.5b-hf


gathering model components 
copying file sha256:86afa7d8ef2e622fb9a917c75008198e791eac225937c687c0910fc6e46f377e 100% 
copying file sha256:62028535ce9a2e3c00d376afc32f7342911a196f3dcb8977b9d4b1933874ca0b 100% 
copying file sha256:3fd169731d2cbde95e10bf356d66d5997fd885dd8dbb6fb4684da3f23b2585d8 100% 
copying file sha256:ee8f6d44bf2353e6d3686c3adaf70e1ccfe9e6ed6822d0ab2f28cafdd7754792 100% 
copying file sha256:dc273a10443fb9490ef35017e37a3f9d4bc7b8641a24c4c5c062cae2356df369 100% 
copying file sha256:d5495a1e5db0611132a97e46a65dbb64a642a499421228b9c8b93229097fa9a4 100% 
converting model 
verifying conversion ⠹ gathering model components 
copying file sha256:86afa7d8ef2e622fb9a917c75008198e791eac225937c687c0910fc6e46f377e 100% 
copying file sha256:62028535ce9a2e3c00d376afc32f7342911a196f3dcb8977b9d4b1933874ca0b 100% 
copying file sha256:3fd169731d2cbde95e10bf356d66d5997fd885dd8dbb6fb4684da3f23b2585d8 100% 
copying file sha256:ee8f6d44bf2353e6d3686c3adaf70e1ccfe9e6ed6822d0ab2f28cafdd7754792

In [26]:
# 8. Smoke-Test Ollama Local Endpoint with Grammar-Constrained Output
import json
import urllib.request

payload = {
    "model": OLLAMA_MODEL_NAME,
    "stream": False,
    "format": {
        "type": "string",
        "enum": VALID_CATEGORIES,
    },
    "messages": [
        {
            "role": "user",
            "content": format_user_prompt(
                "Hotter recho",
                "149.00 debited to VPA hotstaronline@ybl JioHotstar",
                149.0,
            ),
        }
    ],
}

req = urllib.request.Request(
    "http://localhost:11434/api/chat",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
)

with urllib.request.urlopen(req, timeout=15) as resp:
    result = json.loads(resp.read().decode("utf-8"))
    print("Ollama Prediction:", result["message"]["content"])

Ollama Prediction: "Bills"
